# Satellite Imagery Pipeline

## Trabalho prático para disciplina de Processamento de Imagens

> Desenvolvido por Murilo M. Grosso e Octávio X. Fúrio

### Pipeline setup

In [13]:
ALIST_PSIZE = 2304
CHANNEL_SNR = -1     # dB: Realistically, 1dB
MAX_ITERS   = 20    # More are better & slower

In [14]:
from pathlib import Path
import subprocess

# g++ src/*.cpp -o bin/main -I includes/
subprocess.run("g++ src/*.cpp -o ./jpeg -I includes/", shell=True)

Path("1-inputs").mkdir(parents=True, exist_ok=True)
Path("3-outputs").mkdir(parents=True, exist_ok=True)

# Etapas do pipeline
Path("2-pipeline/bitstreams").mkdir(parents=True, exist_ok=True)
Path("2-pipeline/rec_bitstreams").mkdir(parents=True, exist_ok=True)

# Path("metadata").mkdir(parents=True, exist_ok=True)

### Agora, adicione as imagens a serem transmitidas no diretório 1-inputs.

Note que são aceitas apenas imagens png em RGB

### Compress the input images

In [15]:
# Helpers
import numpy as np
import os

def pack_bits(bits, k):
    sentinel = np.concatenate([bits, [1]])
    remainder = len(sentinel) % k
    padding = (k - remainder) % k
    return np.concatenate([sentinel, np.zeros(padding, dtype=np.uint8)]).reshape(-1, k)

def unpack_bits(packets):
    bits = packets.ravel()
    return bits[:np.where(bits)[0][-1]]

In [16]:
# Compression of images
for im in os.listdir("1-inputs"):
    # EXE 0 image_name.png bitstream.data
    subprocess.run(
        f"jpeg.exe 0 1-inputs/{im} 2-pipeline/bitstreams/{im.split('.')[0]}.data",
        shell=True
    )

In [ ]:
# Encoding, transmission, error correction
from src.LDPC import LDPCEncoder, LDPCDecoder, awgn_channel

alist_file = f"./alist/{ALIST_PSIZE}.alist"

encoder = LDPCEncoder(alist_file, 5, 7)
decoder = LDPCDecoder(alist_file)

k = encoder.N - encoder.M

for fname in os.listdir("2-pipeline/bitstreams"):
    with open(os.path.join("2-pipeline/bitstreams", fname), "rb") as f:
        w = np.frombuffer(f.read(4), dtype=np.int32)[0]
        h = np.frombuffer(f.read(4), dtype=np.int32)[0]
        bits = np.unpackbits(
            np.frombuffer(f.read(), dtype=np.uint8)
        )
    packets = pack_bits(bits, k)
    recovered = []
    uncorrected = []

    # Encode & Transmit
    for pkt in packets:
        llr = awgn_channel(encoder.encode(pkt), CHANNEL_SNR)
        uncorrected.append((llr[:k] <= 0).astype(np.uint8))
        recovered.append(decoder.decode(llr, max_iter=MAX_ITERS)[:k])

    # Reconstruct
    out_bits = np.packbits(unpack_bits(np.array(recovered)))
    unc_bits = np.packbits(unpack_bits(np.array(uncorrected)))

    with open(os.path.join("2-pipeline/rec_bitstreams", fname), "wb") as f:
        f.write(np.array([w, h], dtype=np.int32).tobytes())
        f.write(out_bits.tobytes())

    with open(os.path.join("2-pipeline/rec_bitstreams", f"unc_{fname}"), "wb") as f:
        f.write(np.array([w, h], dtype=np.int32).tobytes())
        f.write(unc_bits.tobytes())

    print(f"{fname} done")

House.data done
Op.data done


In [18]:
# Decompression of images
Path(f"3-outputs/SNR{CHANNEL_SNR}dB").mkdir(parents=True, exist_ok=True)

for im in os.listdir("2-pipeline/rec_bitstreams"):
    # EXE 1 bitstream.data image_name.png
    subprocess.run(
        f"jpeg.exe 1 2-pipeline/rec_bitstreams/{im} 3-outputs/SNR{CHANNEL_SNR}dB/{im[:-5]}_{MAX_ITERS}Iters.png",
        shell=True
    )

### Imagens resultantes em 3-outputs